<a href="https://colab.research.google.com/github/192565027simats/CSA6102/blob/main/EXP33-Port%20Scan%20Detector%20from%20Packet%20Metadata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from datetime import datetime, timedelta

PKT_FMT = "%Y-%m-%d %H:%M:%S"

def detect_port_scan(packets, port_threshold=10, window_seconds=30):
    """
    Detect (src_ip -> dst_ip) pairs that contact
    >= port_threshold distinct destination ports
    within window_seconds.
    """
    packets = sorted(
        packets,
        key=lambda p: datetime.strptime(p["timestamp"], PKT_FMT)
    )

    by_pair = {}

    for packet in packets:
        key = (packet["src_ip"], packet["dst_ip"])
        by_pair.setdefault(key, []).append(packet)

    results = {}

    for key, packet_list in by_pair.items():
        for i in range(len(packet_list)):
            start = datetime.strptime(packet_list[i]["timestamp"], PKT_FMT)
            end = start + timedelta(seconds=window_seconds)

            ports = {
                pkt["dst_port"]
                for pkt in packet_list
                if start <= datetime.strptime(pkt["timestamp"], PKT_FMT) <= end
            }

            if len(ports) >= port_threshold:
                results[key] = {
                    "distinct_ports": len(ports),
                    "ports": sorted(ports),
                }
                break

    return results


# ---------------- Sample Input ----------------

packets = []
base = datetime(2026, 3, 1, 10, 0, 0)

# Simulated Port Scan
for i, port in enumerate(range(20, 32)):
    packets.append({
        "src_ip": "203.0.113.99",
        "dst_ip": "10.0.0.10",
        "dst_port": port,
        "timestamp": (base + timedelta(seconds=2 * i)).strftime(PKT_FMT),
    })

# Normal Network Traffic
for i in range(5):
    packets.append({
        "src_ip": "10.0.0.20",
        "dst_ip": "10.0.0.30",
        "dst_port": 443,
        "timestamp": (base + timedelta(seconds=5 * i)).strftime(PKT_FMT),
    })

# ---------------- Run Detection ----------------

results = detect_port_scan(
    packets,
    port_threshold=10,
    window_seconds=30
)

# ---------------- Verification ----------------

assert ("203.0.113.99", "10.0.0.10") in results
assert results[("203.0.113.99", "10.0.0.10")]["distinct_ports"] >= 10
assert ("10.0.0.20", "10.0.0.30") not in results

print("All test cases passed.\n")

print("Detected Port Scans:")
for pair, info in results.items():
    print(f"\nSource IP        : {pair[0]}")
    print(f"Destination IP   : {pair[1]}")
    print(f"Distinct Ports   : {info['distinct_ports']}")
    print(f"Ports Scanned    : {info['ports']}")

All test cases passed.

Detected Port Scans:

Source IP        : 203.0.113.99
Destination IP   : 10.0.0.10
Distinct Ports   : 12
Ports Scanned    : [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
